In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
REPORT_FOLDER = REPO_ROOT / "data_load" / "load_reports"
TOP_1_ONLY = False

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run notebooks 01 to 04 first"

In [ ]:
from cscaf.loading import AdvertisementLoader, AdvertisementSimilarities
from cscaf.storage import Storage
from cscaf.study import Study
from cscaf.text import TextPipeline

study = Study.from_config()
storage = Storage.of(study, read_only=False)
text = TextPipeline(study)
run_id = study.run_id
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)
segmentation = study.section("segmentation")
print("database           :", storage.path.relative_to(REPO_ROOT))
print("segmentation model :", segmentation["model"], "@", segmentation["model_revision"])
print("preprocessing      :", "on" if segmentation.get("preprocessing", {}).get("enabled") else "OFF")
print("line breaks forced :", segmentation["split_on_input_newlines"])
print("embedding model    :", text.model_name, "| stored as", text.embedding_dtype)
print("run                :", run_id)

In [ ]:
content = {
    "rf_sentence": storage.count_rows("rf_sentence", run_id),
    "job_advertisement": storage.count_rows("job_advertisement"),
    "job_advertisement_sentence": storage.count_rows("job_advertisement_sentence", run_id),
    "job_advertisement_rf_similarity_all": storage.similarities.count_advertisement_similarities(run_id),
    "job_advertisement_rf_similarity_top_1": storage.count_rows("job_advertisement_rf_similarity_top_1", run_id),
}
content

In [ ]:
assert content["rf_sentence"], "no ECSF statements for this run: run 01_load_ecsf_to_database.ipynb first"
assert content["job_advertisement"], "no advertisements: run 04_load_job_advertisements_to_database.ipynb first"
if not TOP_1_ONLY:
    segmentation_report = AdvertisementLoader(study, storage, text).segment(REPORT_FOLDER)
    problems = segmentation_report.verify(storage)
    assert not problems, problems
    print("ads without any sentence:", segmentation_report.ads_without_sentences)

In [ ]:
report = AdvertisementSimilarities(storage, run_id).compute(REPORT_FOLDER, TOP_1_ONLY)

In [ ]:
check = report.check
assert check["all_rows"] == check["statements"] * check["sentences"], check
assert check["top_1_rows"] == check["statements"] * check["ads"], check
assert check["not_max"] == 0 and check["wrong_sentence"] == 0, check
assert report.sample["max_abs_difference"] <= report.sample["tolerance"], report.sample
print(check)
print(report.sample)